## Title→raw_id 字符串匹配（双向包裹 + Aho-Corasick）

数据形态（`first_sentence` 固定为段落前 **100** 字符）：

| 情况 | 关系 | 匹配方式 |
|------|------|----------|
| `len(title) ≤ 100` | **title 被 first_sentence 包裹**（title ⊆ fs） | 在 `fs` 里搜 title（AC 多模式） |
| `len(title) > 100` | **first_sentence 被 title 包裹**（fs ⊆ title，通常 fs 是 title 前缀） | 用 title 的前 100 字在 `fs` 里命中后，再校验 `title.startswith(fs)` |

**严格匹配规则（避免“匹配一两个词就算命中”）：**
- 短标题：`first_sentence` 必须以**完整 title** 开头（`fs.startswith(title)`），禁止句中子串。
- 长标题：`fs` 对应 `normalize(title[:100])`（先截断再归一化，与建库 `text[:100]` 一致）；并可用词级前缀容忍第 100 字处截断半词。
- 跳过过短/模板化标题（如 `Introduction` / `Editorial`，长度 < 20 或词数 < 3）。
- 归一化保留 Unicode 字母（避免非英文标题被剥成 `dna` 等碎片误匹配）。
- 每个 title 在全库扫描结束后保留**得分最高**的一条，而不是第一次碰到的行。


In [ ]:
from pathlib import Path

import sys

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from sentence_string_match import (
    DEFAULT_FIRST_SENT_DIR,
    FIRST_SENT_LEN,
    iter_jsonl,
    normalize,
    is_indexable_title,
    search_search_paper_dir,
)

# Agent 目录（含 search_paper.jsonl）
AGENT_DIR = PROJECT_ROOT / "microbiome_ecology_copy"
FIRST_SENT_DIR = DEFAULT_FIRST_SENT_DIR
UNMATCHED_OUT_PATH = AGENT_DIR / "title_unmatched.jsonl"


In [20]:
# 核心逻辑在 sentence_string_match.py（依赖 pyahocorasick）
# CLI: python sentence_string_match.py search --dir microbiome_ecology_copy


In [21]:
results = search_search_paper_dir(
    AGENT_DIR,
    first_sent_dir=FIRST_SENT_DIR,
    output_name="search_results.json",
    unmatched_name="title_unmatched.jsonl",
)
list(results.items())[:3]


In [22]:
import json
p = AGENT_DIR / "search_results.json"
with p.open(encoding="utf-8") as f:
    data = json.load(f)
n = sum(len(v) for v in data.values())
print(f"Shards: {len(data)}, total raw_ids: {n}")


Loaded titles: 10428
Title index: short=4,516, long=5,555, skipped=25 (too short/generic)
Scanning raw_passages-0-of-16.first_sentences.jsonl ...
Scanning raw_passages-1-of-16.first_sentences.jsonl ...
Scanning raw_passages-10-of-16.first_sentences.jsonl ...
Scanning raw_passages-11-of-16.first_sentences.jsonl ...
Scanning raw_passages-12-of-16.first_sentences.jsonl ...
Scanning raw_passages-13-of-16.first_sentences.jsonl ...
Scanning raw_passages-14-of-16.first_sentences.jsonl ...
Scanning raw_passages-15-of-16.first_sentences.jsonl ...
Scanning raw_passages-2-of-16.first_sentences.jsonl ...
Scanning raw_passages-3-of-16.first_sentences.jsonl ...
Scanning raw_passages-4-of-16.first_sentences.jsonl ...
Scanning raw_passages-5-of-16.first_sentences.jsonl ...
Scanning raw_passages-6-of-16.first_sentences.jsonl ...
Scanning raw_passages-7-of-16.first_sentences.jsonl ...
Scanning raw_passages-8-of-16.first_sentences.jsonl ...
Scanning raw_passages-9-of-16.first_sentences.jsonl ...
Stats: {

## 未命中标题 → BM25 粗排（抽样）

从 `title_unmatched.jsonl` 取 5 条，调用 `bm25_db_construct.search_titles` 在 `sentence_bm25_indexes` 上检索。  
需已构建 BM25 索引；首次运行会加载 16 个分片，耗时较长。


In [23]:
import sys

sys.path.insert(0, str(PROJECT_ROOT))

from bm25_db_construct import (
    DEFAULT_INDEX_DIR,
    DEFAULT_META_DIR,
    search_titles,
)

BM25_SAMPLE_N = 5
BM25_TOP_K = 1
BM25_PER_SHARD_K = 5
BM25_N_THREADS = 0      # bm25s.retrieve 线程数，0=自动
BM25_SHARD_WORKERS = 0  # 分片并行数，0=自动


def load_unmatched_sample(
    path: Path,
    n: int = 5,
    *,
    reason: Optional[str] = "no_match_in_corpus",
) -> Tuple[List[str], List[dict]]:
    titles: List[str] = []
    records: List[dict] = []
    for rec in iter_jsonl(path):
        if reason and rec.get("reason") != reason:
            continue
        title = (rec.get("title") or "").strip()
        if not title:
            continue
        titles.append(title)
        records.append(rec)
        if len(titles) >= n:
            break
    return titles, records


if not UNMATCHED_OUT_PATH.exists():
    raise FileNotFoundError(f"Run matching cell first: {UNMATCHED_OUT_PATH}")

sample_titles, sample_records = load_unmatched_sample(UNMATCHED_OUT_PATH, BM25_SAMPLE_N)
print(f"Loaded {len(sample_titles)} titles from {UNMATCHED_OUT_PATH.name}")
print(f"BM25 indexes: {DEFAULT_INDEX_DIR}")

all_hits = search_titles(
    sample_titles,
    index_dir=DEFAULT_INDEX_DIR,
    meta_dir=DEFAULT_META_DIR,
    final_k=BM25_TOP_K,
    per_shard_k=BM25_PER_SHARD_K,
    n_threads=BM25_N_THREADS,
    shard_workers=BM25_SHARD_WORKERS,
)

for i, (rec, hits) in enumerate(zip(sample_records, all_hits), 1):
    print("\n" + "=" * 80)
    print(f"[{i}] paperId={rec.get('paperId')}  reason={rec.get('reason')}")
    title = rec["title"]
    print(f"Query: {title}")
    if not hits:
        print("  (no BM25 hits)")
        continue
    for rank, h in enumerate(hits, 1):
        preview = h.first_sentence
        print(f"  {rank}. score={h.score:.4f}  raw_id={h.raw_id}  shard={h.shard}")
        print(f"      {preview}")


Loaded 5 titles from title_unmatched.jsonl
BM25 indexes: D:\Projects\BotMaker\OpenScholar-DataStore-V3\sentence_bm25_indexes
Loading BM25 index raw_passages-0-of-16 ...
Loading meta raw_passages-0-of-16.first_sentences.raw_id.jsonl ...
Loading BM25 index raw_passages-1-of-16 ...
Loading meta raw_passages-1-of-16.first_sentences.raw_id.jsonl ...
Loading BM25 index raw_passages-10-of-16 ...
Loading meta raw_passages-10-of-16.first_sentences.raw_id.jsonl ...
Loading BM25 index raw_passages-11-of-16 ...
Loading meta raw_passages-11-of-16.first_sentences.raw_id.jsonl ...
Loading BM25 index raw_passages-12-of-16 ...
Loading meta raw_passages-12-of-16.first_sentences.raw_id.jsonl ...
Loading BM25 index raw_passages-13-of-16 ...
Loading meta raw_passages-13-of-16.first_sentences.raw_id.jsonl ...
Loading BM25 index raw_passages-14-of-16 ...
Loading meta raw_passages-14-of-16.first_sentences.raw_id.jsonl ...
Loading BM25 index raw_passages-15-of-16 ...
Loading meta raw_passages-15-of-16.first_se

## 方案对比（你这个数据形态推荐的顺序）

- **最快（但依赖“标题在开头且分隔符稳定”）**：从 `first_sentence` 里抽取“疑似标题片段”（比如遇到两空格/句号/逗号前）→ 归一化 → set 查找。
  - 优点：O(1) 查找，极快
  - 缺点：需要你能稳定抽出标题边界，否则误匹配/漏匹配。

- **通用且快（推荐）**：**Aho‑Corasick 多模式匹配**（本 notebook 采用）。
  - 优点：一次构建自动机，扫描每条 `first_sentence` 只需线性时间，适合“几万标题 + 4000 万文本”的批量匹配。
  - 缺点：需要 `pyahocorasick`（C 扩展）。

- **最通用但慢**：编辑距离/模糊匹配（fuzzywuzzy/rapidfuzz）逐条做。
  - 在 40M 规模上通常不可用，除非你做强约束候选集。
